# CN-GongWen · 基于明确 prompt 的两阶段公文写作（MiniMax）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/SH-PA/blob/claude/determined-lamport-vyvmpd/notebooks/CN_GongWen_Compose_Colab.ipynb)

**实际测试流程**：给定明确的写作 prompt（文种、发文机关、行文方向、事由、长度等）→
**第一步**用 MiniMax 生成「写作框架」(提纲) → **第二步**据框架生成「整体公文」全文。
覆盖**不同文种**（15 种法定公文）与**不同长度**（短/中/长）、不同政策方向（通用政务 / 医疗卫生及其具体主题）。

> 提供 `MINIMAX_API_KEY` 即真正调用 MiniMax 写作；留空则用确定性模板离线演示同样的两阶段结构。


## 1️⃣ 克隆仓库


In [ ]:
import os
BASE = '/content' if os.path.isdir('/content') else os.getcwd()
os.chdir(BASE)
if not os.path.isdir('SH-PA'):
    !git clone --branch claude/determined-lamport-vyvmpd https://github.com/pariskang/SH-PA.git SH-PA
os.chdir(os.path.join(BASE, 'SH-PA'))
!git checkout -q claude/determined-lamport-vyvmpd 2>/dev/null; git pull -q origin claude/determined-lamport-vyvmpd 2>/dev/null
print('CWD:', os.getcwd())
!git log --oneline -2


## 2️⃣ 安装依赖

`openai` 用于调用 OpenAI 兼容的 MiniMax 接口。


In [ ]:
!pip -q install openai pytest
print('依赖安装完成')


## 3️⃣ 配置 MiniMax 凭证

下面是你提供的凭证单元（表单 / Secrets / 弹窗输入）。


In [ ]:
#@title 写入凭证（Secrets / 弹窗输入） { display-mode: "form" }
MINIMAX_MODEL    = "MiniMax-M1" #@param {type:"string"}
MINIMAX_API_BASE = "https://api.minimaxi.com/v1" #@param {type:"string"}
#@markdown > 若用自建 OpenAI 兼容中转，请把 `MINIMAX_API_BASE` 改成你的 base_url。

import os
os.environ["MINIMAX_MODEL"] = MINIMAX_MODEL
os.environ["MINIMAX_API_BASE"] = MINIMAX_API_BASE

key = os.environ.get("MINIMAX_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("MINIMAX_API_KEY")
    except Exception:
        key = None
if not key:
    import getpass
    key = getpass.getpass("输入 MINIMAX_API_KEY（留空=跳过 LLM，使用确定性复现）：").strip()

if key:
    os.environ["MINIMAX_API_KEY"] = key
    print(f"✅ MINIMAX_API_KEY 已配置（{key[:4]}…，长度 {len(key)}）")
    print(f"   base = {MINIMAX_API_BASE}  |  model = {MINIMAX_MODEL}")
else:
    print("⚠️ 未提供 API Key —— 可继续用确定性（无 LLM）复现，隐藏标签/数值完全一致。")


## 4️⃣ 载入两阶段生成器，查看「明确写作要求」一览

下表每行就是一个明确的写作 prompt（不同文种 × 不同长度 × 医疗/通用）。


In [ ]:
import os, json, sys
from gongwen_benchmark.scripts.compose_documents import build_specs, compose, DocSpec, LENGTHS
from gongwen_benchmark.scripts.litellm_minimax import LiteLLMConfig
USE_LLM = bool(os.getenv('MINIMAX_API_KEY'))
cfg = LiteLLMConfig() if USE_LLM else None
print('生成引擎：', 'MiniMax 真实写作' if USE_LLM else '确定性离线回退')
specs = build_specs(6)
for s in specs:
    L = LENGTHS[s.length]['label']
    tag = f"{s.policy_domain}{('/'+s.medical_topic) if s.medical_topic else ''}"
    print(f"[{s.spec_id}] {s.doc_type} · {L} · {tag}")
    print('     ', s.prompt_brief())


## 5️⃣ 两阶段生成：写作框架 → 整体公文


In [ ]:
for s in specs:
    r = compose(s, cfg, USE_LLM)
    print('='*70)
    print(f"【明确写作要求】{s.prompt_brief()}")
    print('\n— 第一步·写作框架 —')
    print(json.dumps(r['framework'], ensure_ascii=False, indent=2))
    print('\n— 第二步·整体公文 —')
    print(r['document'])


## 6️⃣ 自定义你的明确 prompt（即时生成）

修改下方表单，运行即按你的要求两阶段生成一篇公文。


In [ ]:
#@title 自定义明确 prompt → 写作框架 → 整体公文 { display-mode: "form" }
文种 = "通知" #@param ["通知","请示","报告","批复","意见","通报","函","纪要","公告","通告","决定","决议","命令","公报","议案"]
事由 = "推进DRG/DIP医保支付方式改革" #@param {type:"string"}
发文机关 = "示范市医疗保障局" #@param {type:"string"}
行文方向 = "downward" #@param ["downward","upward","parallel"]
长度 = "medium" #@param ["short","medium","long"]
主送机关 = "各定点医疗机构、各县（市、区）医疗保障局" #@param {type:"string"}

from gongwen_benchmark.scripts.benchmark_schema import doc_type_by_name
dt = doc_type_by_name(文种)
is_med = any(k in 事由 for k in ("医","药","疫","健康","诊疗","卫生","病","护"))
spec = DocSpec(spec_id="CUSTOM", doc_type=文种, category=dt.category,
               policy_domain=("医疗卫生" if is_med else "通用政务"), medical_area="", medical_topic="",
               subject=事由, agency=发文机关, recipient=主送机关, direction=行文方向,
               length=长度, security="公开", urgency="平件")
r = compose(spec, cfg, USE_LLM)
print("— 第一步·写作框架 —")
print(json.dumps(r["framework"], ensure_ascii=False, indent=2))
print("\n— 第二步·整体公文 —\n")
print(r["document"])


## 7️⃣ 批量生成并保存

批量覆盖全部 15 文种 × 三种长度，保存到 `gongwen_benchmark/generated_documents/`（LLM 产物不入库）。


In [ ]:
!python gongwen_benchmark/scripts/compose_documents.py --count 15 --out gongwen_benchmark/generated_documents
import glob
print('\n生成文件：')
for p in sorted(glob.glob('gongwen_benchmark/generated_documents/*'))[:8]: print('  ', p)


## ✅ 小结

本脚本演示了**基于明确 prompt 的两阶段公文写作**：写作框架 → 整体公文，覆盖不同文种与不同长度。
- 有 `MINIMAX_API_KEY`：真正调用 MiniMax（`MiniMax-M1`，OpenAI 兼容）写作；
- 无 Key：确定性离线回退，结构一致、可审阅。

确定性样例已提交在 `gongwen_benchmark/generated_samples/`（含 `.md` 与 `documents.jsonl`）。
